# EM-FloatFormer Stage 3 — Ministral-3-3B Cache Alignment & Direct Readout Runner

Runs end-to-end on Google Colab GPU (Zero base 3B weights downloaded or stored locally):
1. Verify GPU, PyTorch & Transformers (>=5.0 / Mistral3 support), and shallow-clone `research/stage3-third-window-ministral`.
2. Run baseline checks + Section A (`verify_theory.py`) + Stage 3 CPU invariants (`pytest`).
3. Run Section B (`eval_direct.py`) across `C1..C6` + `504f3a44` regression + paired bootstrap CIs.
4. Run Section C.1 (`train_connector.py --smoke`) verifying frozen host, gradient isolation, KV-cache parity, and Chinese decoding.
5. Run Section C.2 (`train_connector.py --train`) on `s3_train` + lock calibration & tolerances on `s3_dev` (`host_dev_lock.json`).
6. Run Section C.3 (`eval_host.py`) blind `s3_test` evaluation + C.4 (`chat_demo.py`) replayable Chinese transcript.
7. Package `< 10 MB` deliverables + `MANIFEST.sha256` into `stage3_deliverables.tar.gz` and trigger download.

In [ ]:
# Cell 1: GPU Check, Repo Sync, and Transformers Compatibility
import os, sys, subprocess, importlib
import torch

assert torch.cuda.is_available(), 'GPU runtime required! Switch Runtime -> Change runtime type -> T4/L4/A100 GPU.'
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__, 'BF16 supported:', torch.cuda.is_bf16_supported())
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

check_cmd = [
    sys.executable, '-c',
    "import transformers; from transformers import Mistral3ForConditionalGeneration; "
    "from transformers.models.auto.configuration_auto import CONFIG_MAPPING; "
    "assert 'ministral3' in CONFIG_MAPPING; print('OK:', transformers.__version__)",
]
chk = subprocess.run(check_cmd, capture_output=True, text=True)
if chk.returncode != 0:
    print('Upgrading transformers for Mistral3/ministral3:', chk.stderr.strip())
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'transformers>=5.0.0', 'accelerate', 'huggingface_hub', 'mistral-common', 'sentencepiece', 'protobuf',
    ])
    subprocess.check_call(check_cmd)
else:
    print(chk.stdout.strip())

REPO_DIR = '/content/EM-FloatFormer'
BRANCH = 'research/stage3-third-window-ministral'
REPO_URL = 'https://github.com/20171806011/EM-FloatFormer.git'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'])

os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

In [ ]:
# Cell 2: Baseline Lock Replay + Section A Theory Verification + Stage 3 Pytest Suite
import os, subprocess, sys
os.chdir('/content/EM-FloatFormer')

for cmd in [
    [sys.executable, '-B', 'research/matched_cache/audit.py'],
    [sys.executable, '-B', 'research/matched_cache/physical.py'],
    [sys.executable, '-B', 'research/stage3_ministral_cache/verify_theory.py'],
    [sys.executable, '-B', '-m', 'pytest', 'research/stage3_ministral_cache/tests', '-v'],
]:
    print('>>> Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

In [ ]:
# Cell 3: Section B — Frozen Direct Readout Full Evaluation (C1..C6 + 504f3a44 Regression + Bootstrap CIs)
import json, os, subprocess, sys
os.chdir('/content/EM-FloatFormer')

subprocess.check_call([sys.executable, '-B', 'research/stage3_ministral_cache/eval_direct.py', '--split', 's3_test'])
direct_summary = json.load(open('research/stage3_ministral_cache/results/direct/direct_summary.json', 'r', encoding='utf-8'))
print('R0 Regression pass:', direct_summary['regression_504f3a44']['regression_pass'],
      'max_diff:', direct_summary['regression_504f3a44']['max_abs_diff_vs_504f3a44'])

In [ ]:
# Cell 4: Section C.1 — Frozen Ministral-3-3B Host Smoke Audit on GPU
import json, os, subprocess, sys
os.chdir('/content/EM-FloatFormer')

subprocess.check_call([sys.executable, '-B', 'research/stage3_ministral_cache/train_connector.py', '--smoke'])
smoke = json.load(open('research/stage3_ministral_cache/results/host/smoke_audit.json', 'r', encoding='utf-8'))
assert smoke['host_sha_identical'], 'Host SHA check failed!'
assert smoke['host_grads_nonzero'] == 0, 'Host received gradients!'
assert smoke['conn_grads_all_nonzero'], 'Connector missing gradients!'
print('Smoke OK | Peak VRAM MB:', smoke['peak_vram_mb'], '| Rec test blocks:', smoke['recommended_host_test_blocks'])

In [ ]:
# Cell 5: Section C.2 — Train Small Cache Connectors on s3_train + Lock Tolerances on s3_dev
import json, os, subprocess, sys
os.chdir('/content/EM-FloatFormer')

subprocess.check_call([
    sys.executable, '-B', 'research/stage3_ministral_cache/train_connector.py',
    '--train',
    '--encoders', 'equivariant_b8_s17,hand_b8,plain_b8_s17',
    '--epochs', '4',
    '--batch-size', '32',
    '--lr', '0.003',
    '--train-blocks', '2',
    '--dev-blocks', '2',
    '--max-steps-per-epoch', '120',
])
lock = json.load(open('research/stage3_ministral_cache/results/host/host_dev_lock.json', 'r', encoding='utf-8'))
assert lock['host_sha_identical'], 'Host weights modified during connector training!'
print('Dev lock finalized BEFORE test:', json.dumps(lock, ensure_ascii=False, indent=2))

In [ ]:
# Cell 6: Section C.3 Blind s3_test Evaluation + Section C.4 Replayable Chinese Transcript
import json, os, subprocess, sys
os.chdir('/content/EM-FloatFormer')

subprocess.check_call([sys.executable, '-B', 'research/stage3_ministral_cache/eval_host.py', '--batch-size', '64'])
subprocess.check_call([sys.executable, '-B', 'research/stage3_ministral_cache/chat_demo.py', '--encoder', 'equivariant_b8_s17'])

hs = json.load(open('research/stage3_ministral_cache/results/host/host_summary.json', 'r', encoding='utf-8'))
print('Host evaluation wall s:', hs['total_eval_wall_s'], '| Peak VRAM MB:', hs['peak_vram_mb'])
for cname, cdata in hs['encoders']['equivariant_b8_s17']['conditions'].items():
    t_q2 = hs['controls']['text_only'][cname]['q2_by_arm']['valid']['acc']
    s_q2 = hs['controls']['shuffled_Z'][cname]['q2_by_arm']['valid']['acc']
    print(f"{cname:22s} | Q1={cdata['q1']['acc']:.4f} Q3_rep={cdata['q3_repeat']['acc']:.4f} Q2_val={cdata['q2_by_arm']['valid']['acc']:.4f} "
          f"pass={cdata['all_three_pass']} | raw_dup_tv={cdata['raw_duplicate_probe_mean_tv']:.4f} | text={t_q2:.4f} shuf={s_q2:.4f}")

In [ ]:
# Cell 7: Generate MANIFEST.sha256, Pack <10MB Lightweight Deliverables, and Download
import hashlib, os, tarfile
from pathlib import Path
os.chdir('/content/EM-FloatFormer')

stage3 = Path('research/stage3_ministral_cache')
deliverables = []
deliverables.append(stage3 / 'verify_theory_output.json')
deliverables.extend(sorted((stage3 / 'results' / 'direct').glob('*.json')))
deliverables.extend(sorted((stage3 / 'results' / 'direct' / 'pre_reveal_jsonl').glob('*.jsonl')))
deliverables.extend(sorted((stage3 / 'results' / 'host').glob('*')))
deliverables.extend(sorted((stage3 / 'connector_weights').glob('*.pt')))

manifest_lines = []
for p in deliverables:
    if p.is_file():
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest_lines.append(f'{h}  {p.as_posix()}')
manifest_path = stage3 / 'MANIFEST.sha256'
manifest_path.write_text('\n'.join(manifest_lines) + '\n', encoding='utf-8')
deliverables.append(manifest_path)

tar_path = Path('/content/stage3_deliverables.tar.gz')
with tarfile.open(tar_path, 'w:gz') as tf:
    for p in deliverables:
        if p.is_file():
            tf.add(p, arcname=p.relative_to(stage3).as_posix())

size_mb = tar_path.stat().st_size / (1024 * 1024)
print(f'Packed {len(deliverables)} files into {tar_path} ({size_mb:.3f} MB)')
assert size_mb < 35.0, f'Deliverables archive unexpectedly large: {size_mb:.3f} MB'

try:
    from google.colab import files
    files.download(str(tar_path))
    print('Download triggered for stage3_deliverables.tar.gz')
except Exception as exc:
    print('Colab download helper error:', repr(exc))